# Append web-search blocked domains with API Management

This lab shows how to add organization domains to a Responses API request while preserving the caller's existing blocklist. APIM only applies the merge when a `web_search` tool is present.

For developers familiar with APIM and Foundry. You need an existing APIM service, a Foundry backend or resource endpoint, an existing model deployment that supports `web_search`, Azure CLI authentication, and the repository's Python environment (`uv sync` at the repo root).

1. Load existing resource settings from `.env`.
2. Inspect the policy and prepare routing to the existing backend.
3. Deploy the lab API.
4. Send requests with and without web search and inspect the backend request in APIM tracing.
5. Consume a streaming response and inspect the search-count headers.

See [README.md](README.md) for configuration and policy behavior, and [Microsoft's domain-filtering documentation](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search#domain-filtering) for the request format.

## 1. Load configuration

Run with this lab directory as the working directory. The loader reads the root `.env`, then this lab's `.env`; process variables win. To select another existing file, set `env_file` below or use `LAB_ENV_FILE`.

`BACKEND_ID` / `APIM_BACKEND_ID` reuses an existing APIM Foundry backend and takes precedence over `AZURE_OPENAI_ENDPOINT`. A resource endpoint creates only a backend on existing APIM. Neither option provisions Foundry or a model deployment. Avoid printing configuration dictionaries because they can contain keys.

In [ ]:
from src.lab import load_config, prepare_deployment, deploy, send_response

# Example: env_file = "../secure-responses-api/.env"
env_file = None
config = load_config(env_file)
model = config.get("AZURE_OPENAI_DEPLOYMENT") or config.get("AZURE_OPENAI_DEPLOYMENT_NAME")
if not model:
    raise ValueError("Set AZURE_OPENAI_DEPLOYMENT to an existing deployment supporting web_search.")
print("Configuration loaded; model deployment:", model)


## 2. Inspect the policy and existing backend

The first `<choose>` in [policy.xml](policy.xml) checks `tools` using a preserved copy of the request body. Its branch validates the web-search filters and appends missing domains, preserving existing entries, `allowed_domains`, and unrelated fields.

`web_search_preview` does not support this filtering contract and is left unchanged. Use `web_search` here.

The next cell performs read-only Azure lookups. It fails if a configured resource has been deleted. For backend pools or custom URLs, supply `BACKEND_RESPONSES_PATH` as described in the README.

In [ ]:
prepared = prepare_deployment(config)
print("APIM service:", prepared["parameters"]["apimServiceName"])
print("Existing backend:", prepared["parameters"]["backendId"] or "Create lab backend for configured endpoint")
print("Backend Responses path:", prepared["parameters"]["backendResponsesPath"])
print("Lab Responses URL:", prepared["responses_url"])

## 3. Deploy the dedicated lab API

The next cell deploys the HTTP-only Function proxy, grants its managed identity Foundry access, publishes its code, and configures APIM after the proxy is ready. It also grants APIM's system-assigned identity **Cognitive Services OpenAI User** for direct JSON requests. An endpoint configuration with `AZURE_OPENAI_API_KEY` uses key authentication instead.

Choose an unused `APIM_API_NAME` / `APIM_API_PATH` for the first run. The CLI account needs deployment and role-assignment permissions. Set `AZURE_OPENAI_RESOURCE_ID` for an account in another subscription or with a custom hostname; see the README for backend-pool settings. Role grants may take a few minutes to propagate.

The Function runs on one Linux B1 instance with filesystem host keys and no Azure Storage account. This is experimental; Microsoft documents host storage as required. The plan incurs charges while retained. See [proxy configuration](README.md#proxy-deployment-and-configuration) for limits.


In [ ]:
deployment = deploy(config, prepared)
print("Deployment state:", deployment["properties"]["provisioningState"])
print("Responses URL:", prepared["responses_url"])

## 4. Control request: no web-search tool

This makes a model request through APIM with no `tools` property. Blocklist validation and `set-body` are skipped. In the portal Test tab, enable tracing and verify that the backend request has no added `tools` or `filters` properties.

In [ ]:
def show_text(response):
    for item in response.get("output", []):
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                print(content["text"])

without_web_search = {"model": model, "input": "Reply with the word hello.", "store": False}
control_response = send_response(config, prepared, without_web_search)
show_text(control_response)

## 5. Web search without an existing blocklist

Only the tool declaration is sent by the client. APIM adds `filters.blocked_domains` with the 11 organization domains. `tool_choice` requests a web-search call, and `include` asks Foundry to return consulted sources.

In [ ]:
with_web_search = {
    "model": model,
    "input": "Search for Azure API Management announcements and summarize one with a source link.",
    "tools": [{"type": "web_search"}],
    "tool_choice": {"type": "web_search"},
    "include": ["web_search_call.action.sources"],
    "store": False,
}
search_response = send_response(config, prepared, with_web_search)
show_text(search_response)

## 6. Preserve the caller's blocklist and other filters

The client blocks `example.com` and `youtube.com`. The gateway must preserve `example.com`, keep a single `youtube.com`, and append the other 10 domains. It must also retain the caller's `allowed_domains` and `search_context_size`.

Run this request, then repeat it in APIM's portal Test tab with tracing enabled. Check the **Backend request body**, rather than relying on the response to echo tool settings. The configured domains are listed in the README.

In [ ]:
from copy import deepcopy

with_existing_blocklist = deepcopy(with_web_search)
with_existing_blocklist["tools"][0].update({
    "search_context_size": "low",
    "filters": {
        "allowed_domains": ["learn.microsoft.com", "azure.microsoft.com"],
        "blocked_domains": ["example.com", "youtube.com"],
    },
})
merged_response = send_response(config, prepared, with_existing_blocklist)
show_text(merged_response)

# This is the client-side request. APIM adds domains only after receiving it.
assert with_existing_blocklist["tools"][0]["filters"]["blocked_domains"] == ["example.com", "youtube.com"]

## 7. Inspect returned search sources

Sources provide a useful observation of the search result. They do not prove that APIM transformed every request correctly; use APIM tracing for that.

In [ ]:
for item in merged_response.get("output", []):
    if item.get("type") == "web_search_call":
        for source in item.get("action", {}).get("sources", []):
            print(source.get("url", ""))

## 8. Stream through the Function and inspect search accounting

Run sections 1–3 first. APIM and the Function relay SSE chunks as they arrive. Initial headers contain `x-web-search-count-status: deferred` and `x-web-search-request-id`. After forwarding the final events, the Function submits the count to the protected APIM reporting API; that request logs `x-web-search-count` with status `proxy-reported` and the same request ID.

Reports retry up to three times within 15 seconds. This may delay HTTP EOF but does not hold back model events. Reports can be lost after a process crash or sustained reporting failure. Use [the logging query](README.md#log-the-headers-in-apim) to correlate records and deduplicate retries.

The example prints early events, search progress, and text deltas. High reasoning effort can delay text even while SSE events are flowing. Set `use_web_search = False` for a comparison without search. The final response is available as `streamed_response`; its local count is only a comparison with the proxy's record. The notebook does not submit usage reports.


In [ ]:
import json
from time import perf_counter
import requests

use_web_search = True
streaming_request = {
    "model": model,
    "input": "Generate a list of the 10 best kids viral toys and summarize with source links.",
    "stream": True,
    "store": False,
    "reasoning": {"effort": "high"},
}
if use_web_search:
    streaming_request.update({
        "tools": [{"type": "web_search"}],
        "tool_choice": {"type": "web_search"},
        "include": ["web_search_call.action.sources"],
    })
else:
    streaming_request["input"] = "Explain the role of an API gateway in three short sentences."

subscription_key = config.get("APIM_SUBSCRIPTION_KEY")
if not subscription_key:
    raise ValueError("Set APIM_SUBSCRIPTION_KEY to an all-APIs or lab-API subscription key.")

started = perf_counter()
first_event_at = None
first_text_at = None
streamed_response = None
with requests.post(
    prepared["responses_url"],
    headers={"api-key": subscription_key, "Accept": "text/event-stream"},
    json=streaming_request, stream=True, timeout=(10, 180),
) as http_response:
    if not http_response.ok:
        detail = http_response.text
        for secret in (subscription_key, config.get("AZURE_OPENAI_API_KEY")):
            if secret:
                detail = detail.replace(secret, "<redacted>")
        raise RuntimeError(f"HTTP {http_response.status_code}: {detail[:2000]}")
    if "text/event-stream" not in http_response.headers.get("Content-Type", "").lower():
        raise RuntimeError("Expected an SSE response; check that the backend supports stream=True.")

    print(f"Response headers received after {perf_counter() - started:.2f}s")
    print("Initial count:", http_response.headers.get("x-web-search-count", "reported separately after completion"))
    print("Count status:", http_response.headers.get("x-web-search-count-status", "missing"))
    search_request_id = http_response.headers.get("x-web-search-request-id")
    print("Search accounting request ID:", search_request_id)
    print()

    # Foundry Responses events carry one JSON object per SSE data line.
    # A small read size avoids adding client-side buffering to ordinary streams.
    for line in http_response.iter_lines(chunk_size=1):
        if not line.startswith(b"data:"):
            continue  # Ignore event names, blank lines, and keepalive comments.
        data = line[5:].strip()
        if data == b"[DONE]":
            break
        if not data:
            continue
        event = json.loads(data)
        event_type = event.get("type")
        if first_event_at is None:
            first_event_at = perf_counter() - started
            print(f"First SSE event ({event_type}) after {first_event_at:.2f}s", flush=True)
        if event_type in ("response.web_search_call.in_progress", "response.web_search_call.searching", "response.web_search_call.completed"):
            print(f"[{perf_counter() - started:.2f}s] {event_type}", flush=True)
        if event_type == "response.output_text.delta":
            if first_text_at is None:
                first_text_at = perf_counter() - started
            print(event.get("delta", ""), end="", flush=True)
        elif event_type in ("response.completed", "response.incomplete", "response.failed"):
            streamed_response = event["response"]
        elif event_type == "error":
            raise RuntimeError(f"Streaming error: {event.get('message', event)}")

print()
if streamed_response is None:
    raise RuntimeError("The stream ended without a terminal response event; it may have been interrupted.")
print("Final status:", streamed_response.get("status"))
if first_text_at is not None:
    print(f"First text received after {first_text_at:.2f}s")
print(f"Total elapsed time: {perf_counter() - started:.2f}s")
if streamed_response.get("error"):
    print("Response error:", streamed_response["error"])
if streamed_response.get("incomplete_details"):
    print("Incomplete details:", streamed_response["incomplete_details"])

from proxy.accounting import count_searches
print("Searches in final event (local comparison):", count_searches(streamed_response))
print("The proxy reports this count directly to APIM after forwarding the final events. Correlate using:", search_request_id)
